# ระดับ 3a — เตรียมข้อมูล FF++ c23 (frame extraction + face crop)

**Paper:** Guo et al., *Revisiting Face Forgery Detection: From Facial Representation to Forgery Detection*, IEEE TPAMI 2026
**Code:** https://github.com/zhenglab/FFDBackbone

**เป้าหมาย:** เตรียมข้อมูล FF++ c23 ให้ตรงกับที่ paper/repo ใช้ เพื่อนำไป fine-tune ในระดับ 3b

| ขั้นตอนตาม paper (Sec. IV-A2) | ที่ทำใน notebook นี้ |
|---|---|
| sample 8 frames at equal intervals | `step = N // 8` แล้วเอา frame `0, step, 2·step, …` 8 ตัวแรก (เหมือน `get_sampled_idx` ของ repo) |
| RetinaFace for face cropping | RetinaFace mobilenet0.25 (FTCN) จาก `face_process/lib/ct/detection` |
| resize cropped faces to 224×224 | align 5 จุด (ArcFace template) → `crop_face_sbi(phase='test')` → resize 224×224 |

**official split ของ FF++** (ไฟล์ `splits/{train,val,test}.json` จาก repo ต้นทาง ondyari/FaceForensics)

| split | identity pair | วิดีโอจริง | วิดีโอปลอม (4 method) | รวม |
|---|---|---|---|---|
| train | 360 | 720 | 2,880 | 3,600 |
| val | 70 | 140 | 560 | 700 |
| test | 70 | 140 | 560 | 700 |

split นี้แบ่งตาม **identity** — คนที่อยู่ใน train จะไม่โผล่ใน val/test (notebook ตรวจให้ในหัวข้อ 4)

**ผลลัพธ์:** โครงสร้างโฟลเดอร์แบบเดียวกับที่ `datasets/Image_dataset.py` ของ repo คาดหวัง

```
ffpp_c23_faces/FF++/face_v1/
    splits/{train,val,test}.json
    original_sequences/youtube/c23/videos/<id>/frame/<frame_idx>.png
    manipulated_sequences/<Method>/c23/videos/<a>_<b>/frame/<frame_idx>.png
```

**ตั้งค่า Kaggle ก่อนรัน**
- Accelerator = **GPU T4 ×2** (ใช้แค่ 1 ตัว แต่ T4×2 ได้ quota เท่ากัน) · Internet = **On**
- Add Input: dataset FF++ c23 (ดูหัวข้อ 3) — notebook ตรวจโครงสร้างให้อัตโนมัติ
- รันแบบ **Save & Run All (Commit)** เพื่อให้ output ถูกเก็บเป็น Kaggle Dataset
- เวลาโดยประมาณ **1.5–3 ชม.** สำหรับ 5,000 วิดีโอ

## 0. ตั้งค่า

In [ ]:
import os, sys, re, time, json, glob, random, shutil, tarfile, subprocess, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
T_START = time.time()

ON_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR  = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("FFD_WORK", "./work")).resolve()
TMP_DIR   = Path("/tmp/ffd") if ON_KAGGLE else WORK_DIR / "tmp"
INPUT_DIR = Path("/kaggle/input") if ON_KAGGLE else Path(os.environ.get("FFD_INPUT", str(WORK_DIR / "input")))
REPO_DIR  = TMP_DIR / "FFDBackbone"
OUT_DIR   = WORK_DIR / "results_level3a"          # รายงาน / csv / รูปตัวอย่าง
for d in [TMP_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

REPO_URL  = "https://github.com/zhenglab/FFDBackbone.git"
SPLITS_URL = "https://raw.githubusercontent.com/ondyari/FaceForensics/master/dataset/splits"

# --- ที่เก็บ crop ---
# "tar"   = เขียน png ลง /tmp แล้วแพ็กเป็น .tar ใน /kaggle/working  (แนะนำ: commit เร็วกว่ามาก)
# "files" = เขียน png ลง /kaggle/working ตรง ๆ (ไฟล์ 40,000 ไฟล์ → commit ช้ามาก)
SAVE_MODE  = "tar"
DATASET_NAME = "ffpp_c23_faces"
FACE_ROOT = (TMP_DIR if SAVE_MODE == "tar" else WORK_DIR) / DATASET_NAME / "FF++" / "face_v1"

# --- ขอบเขตงาน ---
COMPRESSION = "c23"
FF_METHODS  = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]   # 4 method ตาม paper
DO_REAL     = True
DO_METHODS  = FF_METHODS          # ลดเหลือบาง method ได้ถ้าอยากแบ่งรันหลาย session
DO_SPLITS   = ["train", "val", "test"]
LIMIT_VIDEOS = None               # ใส่ตัวเลขเพื่อทดลองรันสั้น ๆ เช่น 20

# --- preprocessing (ค่าตาม repo) ---
N_FRAMES   = 8                    # num_segments ใน DBBF.yaml
IMG_SIZE   = 224
DET_THRES  = 0.5                  # threshold ของ RetinaFace ใน get_valid_faces()
MAX_SHIFT  = 10                   # ถ้า frame ที่เลือกหาหน้าไม่เจอ ให้เลื่อนไปได้สูงสุดกี่ frame
CROP_EXT   = ".png"               # lossless — repo ใช้ .png และ dataloader มองหา .png
SEED = 1234                       # seed เดียวกับ DBBF.yaml

random.seed(SEED)
print("ON_KAGGLE:", ON_KAGGLE, "| FACE_ROOT:", FACE_ROOT, "| SAVE_MODE:", SAVE_MODE)

## 1. ติดตั้ง library
ไม่ใช้ `requirements.txt` ของ repo ตรง ๆ เพราะ pin เวอร์ชันเก่า (numpy 1.23, torch 1.13, timm 0.9.2)
ซึ่งจะทำให้ environment ของ Kaggle พัง — ใช้ของที่ Kaggle มีอยู่แล้ว แล้ว patch โค้ด repo แทน (cell ถัดไป)

In [ ]:
import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ENV = {"python": sys.version.split()[0], "torch": torch.__version__,
       "numpy": np.__version__, "opencv": cv2.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print(ENV)

## 2. Clone repo + patch
Patch เดียวกับระดับ 1–2 (ไม่แตะ logic ของโมเดล):
- `np.float` / `np.int` / `np.bool` → `np.float64` / `np.int64` / `np.bool_` (ถูกลบตั้งแต่ numpy 1.24)
- `np.reshape(..., newshape=...)` → positional (เลิกใช้ใน numpy 2.x)
- เพิ่ม `import torch.utils.model_zoo` ใน RetinaFace loader และโหลด weight ลง CPU ก่อน

In [ ]:
def find_repo_in_input():
    """หา FFDBackbone ที่ Add Input ไว้ (โหมดออฟไลน์) — ค้นแค่ 2 ชั้นแรก ไม่ไล่ทั้ง dataset"""
    if not INPUT_DIR.exists():
        return None
    marker = Path("models") / "lib" / "BEiT_v2" / "modeling_finetune.py"
    for d in sorted(p for p in INPUT_DIR.iterdir() if p.is_dir()):
        cands = [d] + sorted(p for p in d.iterdir() if p.is_dir())[:50]
        for c in cands:
            if (c / marker).exists():
                return c
    return None

def get_repo():
    """1) ที่ clone ไว้แล้ว  2) จาก /kaggle/input (ออฟไลน์)  3) git clone"""
    if (REPO_DIR / "models" / "lib" / "BEiT_v2" / "modeling_finetune.py").exists():
        return "cached"
    src = find_repo_in_input()
    if src is not None:
        print("พบ repo ใน Add Input (โหมดออฟไลน์):", src)
        shutil.copytree(src, REPO_DIR, dirs_exist_ok=True)
        return "input"
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(
            "clone repo ไม่สำเร็จ:\n  " + (r.stderr or "").strip() + "\n\n"
            "ถ้าขึ้น 'Could not resolve host' แปลว่า Internet ปิดอยู่ — แก้ได้ 2 ทาง\n"
            "  (ก) เปิดเน็ต: แผงขวาของ Kaggle -> Notebook options -> Internet = On\n"
            "      (ต้องยืนยันเบอร์โทรที่ kaggle.com/settings ก่อน ถ้ายังไม่เคยทำ)\n"
            "  (ข) โหมดออฟไลน์: ดาวน์โหลด zip ของ repo จาก\n"
            "      https://github.com/zhenglab/FFDBackbone/archive/refs/heads/main.zip\n"
            "      -> อัปเป็น Kaggle Dataset -> Add Input  (cell นี้จะหาเจอเอง)")
    return "clone"

REPO_SOURCE = get_repo()
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
                             capture_output=True, text=True).stdout.strip() or "(จาก Add Input — ไม่มี .git)"
print("repo commit:", REPO_COMMIT)

def patch_repo(repo: Path):
    n = 0
    for py in list((repo / "face_process").rglob("*.py")) + list((repo / "models" / "lib" / "BEiT_v2").rglob("*.py")):
        src = py.read_text(encoding="utf-8", errors="ignore")
        new = re.sub(r"np\.float\b", "np.float64", src)
        new = re.sub(r"np\.int\b", "np.int64", new)
        new = re.sub(r"np\.bool\b", "np.bool_", new)
        new = new.replace("np.reshape(b, newshape=(1, 2))", "np.reshape(b, (1, 2))")
        if py.name == "alignment.py":
            if "import torch.utils.model_zoo" not in new:
                new = "import torch.utils.model_zoo\n" + new
            new = new.replace("model_zoo.load_url(url)", "model_zoo.load_url(url, map_location='cpu')")
        if new != src:
            py.write_text(new, encoding="utf-8")
            n += 1
    return n

print("patched files:", patch_repo(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from face_process.lib.ct.detection import FaceDetector    # RetinaFace (mobilenet0.25) แบบ FTCN
from face_process import face_utils                        # crop_aligned (ArcFace 5-point alignment)
print("imported RetinaFace + face_utils")

## 3. ค้นหาและตรวจสอบ FF++ c23 ใน `/kaggle/input`

Kaggle dataset ของ FF++ แต่ละตัววางโฟลเดอร์ไม่เหมือนกัน (บางตัวเป็น `FaceForensics++_C23/Deepfakes/`,
บางตัวเป็น `manipulated_sequences/Deepfakes/c23/videos/`) — cell นี้จึง **ไม่เดาโครงสร้าง** แต่ใช้
**ชื่อไฟล์** ซึ่งเป็นมาตรฐานของ FF++ เองในการแยกประเภท:

- วิดีโอจริง: `000.mp4` … `999.mp4` (เลข 3 หลัก)
- วิดีโอปลอม: `<target>_<source>.mp4` เช่น `000_003.mp4` — method ดูจากชื่อโฟลเดอร์ที่อยู่

โฟลเดอร์ `DeepFakeDetection` / `actors` (คือชุด DFD ซึ่งเป็นคนละ subset) จะถูกข้าม เพราะ paper ใช้แค่ 4 method

> **Dataset ที่ใช้ได้** (เลือกตัวใดตัวหนึ่งแล้ว *Add Input*)
> - `xdxd003/ff-c23` — FaceForensics++ Dataset (C23), ~17.9 GB
> - `ahmedelbanby/faceforensicsplusplus-c23-deepfakebench-structure` — FF++ C23 (DeepfakeBench-ready)
>
> ถ้า cell นี้บอกว่าไฟล์ไม่ครบ ให้ดูรายงานที่พิมพ์ออกมาว่าขาด method ไหน แล้วค่อย Add Input เพิ่ม

In [ ]:
REAL_RE = re.compile(r"^(\d{3})\.mp4$", re.I)
FAKE_RE = re.compile(r"^(\d{3})_(\d{3})\.mp4$", re.I)
SKIP_KEYS = ("deepfakedetection", "/actors/", "actors_")   # DFD subset — ไม่ใช่ 4 method ของ paper

def scan_input(roots):
    """คืน (real_index, fake_index, report) โดยดูจากชื่อไฟล์ ไม่ขึ้นกับโครงสร้างโฟลเดอร์"""
    real_index, fake_index = {}, {m: {} for m in FF_METHODS}
    seen_dirs = {}
    for root in roots:
        if not Path(root).exists():
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            mp4s = [f for f in filenames if f.lower().endswith(".mp4")]
            if not mp4s:
                continue
            low = (dirpath + "/").replace("\\", "/").lower()
            if any(k in low for k in SKIP_KEYS):
                continue
            # method ของโฟลเดอร์นี้ (ถ้ามี)
            method = next((m for m in FF_METHODS if m.lower() in low), None)
            n_r = n_f = 0
            for f in mp4s:
                p = os.path.join(dirpath, f)
                if REAL_RE.match(f):
                    real_index.setdefault(REAL_RE.match(f).group(1), p); n_r += 1
                elif FAKE_RE.match(f) and method:
                    a, b = FAKE_RE.match(f).groups()
                    fake_index[method].setdefault(f"{a}_{b}", p); n_f += 1
            if n_r or n_f:
                seen_dirs[dirpath] = {"method": method or "(real)", "n_real": n_r, "n_fake": n_f}
    return real_index, fake_index, seen_dirs

real_index, fake_index, seen_dirs = scan_input([INPUT_DIR])
print(f"โฟลเดอร์ที่พบวิดีโอ FF++ ({len(seen_dirs)}):")
for d, info in sorted(seen_dirs.items()):
    print(f"  [{info['method']:<16}] real={info['n_real']:>5} fake={info['n_fake']:>5}  {d}")

print(f"\nวิดีโอจริง (original_sequences/youtube): {len(real_index)} / 1000")
for m in FF_METHODS:
    print(f"วิดีโอปลอม {m:<16}: {len(fake_index[m])} / 1000")

if not real_index:
    raise RuntimeError(
        "ไม่พบวิดีโอจริงของ FF++ (ชื่อไฟล์แบบ 000.mp4) ใน /kaggle/input\\n"
        "→ Add Input dataset ที่มี FF++ c23 แล้วรันใหม่ (ดูรายชื่อ dataset ในหัวข้อ 3)")

## 4. Official split ของ FF++ + ตรวจ identity leakage
`dataloader_util.get_FF_list()` ของ repo อ่าน `splits/{split}.json` ซึ่งเป็น list ของคู่ `[target, source]`
แล้วสร้างรายการ:
- **real** = `original_sequences/youtube/c23/videos/{target}` และ `{source}` ของทุกคู่
- **fake** = `manipulated_sequences/{method}/c23/videos/{target}_{source}` และ `{source}_{target}` ของทุก method

cell นี้ทำตามนั้นทุกบรรทัด แล้วตรวจเพิ่มว่า identity ไม่รั่วข้าม split

In [ ]:
import urllib.request

SPLIT_DIR = TMP_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
splits = {}
for s in ["train", "val", "test"]:
    p = SPLIT_DIR / f"{s}.json"
    if not p.exists():
        urllib.request.urlretrieve(f"{SPLITS_URL}/{s}.json", p)
    splits[s] = json.load(open(p))
    print(f"{s:<5} {len(splits[s])} identity pairs")

# --- ตรวจ identity leakage ---
ids = {s: set(x for pair in splits[s] for x in pair) for s in splits}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    ov = ids[a] & ids[b]
    print(f"identity ซ้ำ {a}-{b}: {len(ov)}" + ("  ✓" if not ov else f"  ✗ {sorted(ov)[:5]}"))
assert not (ids["train"] & ids["val"]) and not (ids["train"] & ids["test"]) and not (ids["val"] & ids["test"])
print("identity ทั้งหมด:", len(ids["train"] | ids["val"] | ids["test"]), "(ควรเป็น 1000)")

REAL_LABEL, FAKE_LABEL = 0, 1
REAL_REL = f"original_sequences/youtube/{COMPRESSION}/videos"
FAKE_REL = lambda m: f"manipulated_sequences/{m}/{COMPRESSION}/videos"

def build_list(split):
    """เหมือน get_FF_list(root, split, compress='c23', only_real=False)"""
    rows = []
    real_names, fake_names = [], []
    for t, s in splits[split]:
        real_names += [t, s]
        fake_names += [f"{t}_{s}", f"{s}_{t}"]
    if DO_REAL:
        for x in real_names:
            rows.append({"split": split, "method": "original", "video": x,
                         "rel": f"{REAL_REL}/{x}", "label": REAL_LABEL,
                         "src": real_index.get(x)})
    for m in DO_METHODS:
        for x in fake_names:
            rows.append({"split": split, "method": m, "video": x,
                         "rel": f"{FAKE_REL(m)}/{x}", "label": FAKE_LABEL,
                         "src": fake_index[m].get(x)})
    return rows

videos = pd.DataFrame([r for s in DO_SPLITS for r in build_list(s)])
videos["found"] = videos["src"].notna()

print("\\nจำนวนวิดีโอที่ต้องใช้ (ตาม official split) และที่หาไฟล์เจอ:")
piv = videos.pivot_table(index="split", columns="method", values="found",
                         aggfunc=["size", "sum"], fill_value=0)
print(piv.to_string())
missing = videos[~videos["found"]]
if len(missing):
    print(f"\\n[WARN] หาไฟล์ไม่เจอ {len(missing)} วิดีโอ เช่น:")
    print(missing.groupby(["split", "method"]).size().to_string())
    print(missing[["split", "method", "video"]].head(10).to_string(index=False))
    print("→ จะข้ามวิดีโอเหล่านี้ (ควร Add Input ให้ครบก่อนเทรนจริง)")

videos = videos[videos["found"]].reset_index(drop=True)
if LIMIT_VIDEOS:
    videos = videos.groupby(["split", "method"], group_keys=False).head(max(1, LIMIT_VIDEOS // 10)).reset_index(drop=True)
    print(f"\\n[TEST MODE] LIMIT_VIDEOS={LIMIT_VIDEOS} → ใช้ {len(videos)} วิดีโอ")
print("\\nรวมวิดีโอที่จะประมวลผล:", len(videos))
videos.to_csv(OUT_DIR / "video_list.csv", index=False)

## 5. Frame sampling + face crop

**เลือก frame** — เหมือน `get_sampled_idx()` ของ repo: `step = N // 8` แล้วใช้ `0, step, 2·step, …` 8 ตัวแรก

**crop แต่ละ frame** — pipeline เดียวกับ `face_process/real_face_process_Frame.py`
1. RetinaFace หาใบหน้า เก็บเฉพาะ score ≥ 0.5
2. frame แรกเลือกหน้าที่ใหญ่สุด, frame ถัดไปเลือกหน้าที่ IoU กับ frame ก่อนหน้าสูงสุด (tracking อย่างง่าย)
3. align ด้วย 5 landmark → `crop_face_sbi(margin=False, crop_by_bbox=True, phase='test')` → resize 224×224
4. บันทึกเป็น PNG (lossless) ที่ `.../<video>/frame/<frame_idx>.png`

**ความต่างจากของผู้เขียน (ต้องระบุในรายงาน):** ผู้เขียนใช้ pipeline แบบ FTCN คือตรวจหน้า**ทุก frame**
แล้ว track หน้าที่ปรากฏนานที่สุด ก่อนจะเลือก 8 frame (repo ไม่ได้แนบสคริปต์ส่วนนี้มา) — notebook นี้
ตรวจเฉพาะ 8 frame ที่เลือก ซึ่งให้ผลเท่ากันกับวิดีโอหน้าเดียวอย่าง FF++ แต่ต่างกันได้ในวิดีโอหลายหน้า

In [ ]:
detector = FaceDetector(0 if DEVICE.type == "cuda" else -1)

def valid_faces(faces, max_count=10, thres=DET_THRES):
    """เหมือน get_valid_faces() ของ repo สำหรับภาพเดียว"""
    out = []
    for face in faces[:max_count]:
        box, lm, score = face
        if score < thres:
            continue
        out.append((box.astype(np.float64), lm.astype(np.float64), float(score)))
    return out

def crop_face_test(img, bbox):
    """crop_face_sbi(..., margin=False, crop_by_bbox=True, phase='test') ของ repo
    margin = w/4 * 0.5 ต่อด้าน"""
    H, W = img.shape[:2]
    (x0, y0), (x1, y1) = bbox
    w, h = x1 - x0, y1 - y0
    w0 = w1 = w / 4 * 0.5
    h0 = h1 = h / 4 * 0.5
    y0n, y1n = max(0, int(y0 - h0)), min(H, int(y1 + h1) + 1)
    x0n, x1n = max(0, int(x0 - w0)), min(W, int(x1 + w1) + 1)
    return img[y0n:y1n, x0n:x1n]

def align_and_crop(img_bgr, face):
    bbox, lm5, _ = face
    aligned, _, bb = face_utils.crop_aligned(img_bgr, lm5, landmarks_68=None,
                                            bboxes=bbox, aligned_image_size=IMG_SIZE)
    crop = crop_face_test(aligned, np.array([[bb[0], bb[1]], [bb[2], bb[3]]]))
    if crop.size == 0:
        return None
    return cv2.resize(crop, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

def box_iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x1 - x0) * max(0.0, y1 - y0)
    area = lambda r: max(0.0, r[2] - r[0]) * max(0.0, r[3] - r[1])
    return inter / (area(a) + area(b) - inter + 1e-9)

def pick_face(faces, prev_box):
    if prev_box is None:
        return max(faces, key=lambda f: (f[0][2] - f[0][0]) * (f[0][3] - f[0][1]))
    return max(faces, key=lambda f: box_iou(f[0], prev_box))

def sample_indices(n_frames, k=N_FRAMES):
    """เหมือน get_sampled_idx() ของ repo"""
    idx = np.arange(n_frames)
    if n_frames >= k:
        step = n_frames // k
        return idx[::step][:k]
    return np.sort(np.resize(idx, k))

def process_video(video_path, out_dir):
    """อ่านวิดีโอ → เลือก 8 frame → crop หน้า → เขียน PNG. คืน (n_saved, info)"""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0, {"error": "open_failed"}
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    if n_total <= 0:
        cap.release()
        return 0, {"error": "no_frames"}
    wanted = sample_indices(n_total)
    (out_dir / "frame").mkdir(parents=True, exist_ok=True)

    n_saved, n_shift, prev_box = 0, 0, None
    for target in wanted:
        saved = False
        for shift in range(MAX_SHIFT + 1):
            fi = int(target) + shift
            if fi >= n_total:
                break
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok or frame is None:
                continue
            try:
                faces = valid_faces(detector.detect(frame))
            except Exception:
                continue
            if not faces:
                continue
            face = pick_face(faces, prev_box)
            crop = align_and_crop(frame, face)
            if crop is None:
                continue
            cv2.imwrite(str(out_dir / "frame" / f"{fi}{CROP_EXT}"), crop)
            prev_box = face[0]
            n_saved += 1
            n_shift += shift
            saved = True
            break
        if not saved:
            pass
    cap.release()
    return n_saved, {"n_total_frames": n_total, "width": W, "height": H, "shift_sum": n_shift}

### 5.1 รันทั้ง dataset (รองรับ resume — รันซ้ำจะข้ามวิดีโอที่ทำแล้ว)

In [ ]:
FACE_ROOT.mkdir(parents=True, exist_ok=True)
progress_path = OUT_DIR / "crop_progress.csv"
done = {}
if progress_path.exists():
    prev = pd.read_csv(progress_path)
    done = dict(zip(prev["rel"], prev["n_saved"]))
    print("พบ progress เดิม:", len(done), "วิดีโอ")

rows, t0 = [], time.time()
pbar = tqdm(videos.itertuples(index=False), total=len(videos), desc="crop faces")
for r in pbar:
    out_dir = FACE_ROOT / r.rel
    if r.rel in done:
        rows.append({"rel": r.rel, "split": r.split, "method": r.method, "video": r.video,
                     "label": r.label, "n_saved": done[r.rel], "resumed": True})
        continue
    if (out_dir / "frame").exists() and len(list((out_dir / "frame").glob(f"*{CROP_EXT}"))) >= N_FRAMES:
        n = len(list((out_dir / "frame").glob(f"*{CROP_EXT}")))
        info = {}
    else:
        try:
            n, info = process_video(r.src, out_dir)
        except Exception as e:
            n, info = 0, {"error": f"{type(e).__name__}: {e}"}
    rows.append({"rel": r.rel, "split": r.split, "method": r.method, "video": r.video,
                 "label": r.label, "n_saved": n, "resumed": False, **info})
    if len(rows) % 200 == 0:
        pd.DataFrame(rows).to_csv(progress_path, index=False)
        pbar.set_postfix(min=f"{(time.time()-t0)/60:.0f}")

crop_log = pd.DataFrame(rows)
crop_log.to_csv(progress_path, index=False)
print(f"\nเสร็จใน {(time.time()-t0)/60:.1f} นาที")

## 6. รายงานคุณภาพการ crop

In [ ]:
crop_log["ok"] = crop_log["n_saved"] >= N_FRAMES
summary = (crop_log.groupby(["split", "method"])
           .agg(n_videos=("ok", "size"), n_ok=("ok", "sum"),
                frames=("n_saved", "sum"), mean_frames=("n_saved", "mean"))
           .reset_index())
summary["ok_rate"] = (100 * summary["n_ok"] / summary["n_videos"]).round(2)
print(summary.to_string(index=False))
summary.to_csv(OUT_DIR / "crop_summary.csv", index=False)

bad = crop_log[~crop_log["ok"]]
print(f"\nวิดีโอที่ได้ frame ไม่ครบ {N_FRAMES}: {len(bad)} / {len(crop_log)}")
if len(bad):
    print(bad["n_saved"].value_counts().sort_index().to_string())
    print("\nตัวอย่าง:", bad[["rel", "n_saved"]].head(5).to_string(index=False))
print(f"\nframe ทั้งหมดที่ได้: {int(crop_log['n_saved'].sum()):,}")

# เทียบจำนวนที่ควรได้ตาม official split
exp = {"train": 3600, "val": 700, "test": 700}
for s in DO_SPLITS:
    got = int(crop_log[crop_log["split"] == s]["ok"].sum())
    print(f"{s:<5} วิดีโอที่ใช้ได้ {got} / {exp[s]} (ครบทุก method)")

### 6.1 ภาพตัวอย่างหน้าที่ crop ได้ (คู่ real/fake จาก identity เดียวกัน)

In [ ]:
ok = crop_log[crop_log["ok"]]
show_methods = ["original"] + [m for m in DO_METHODS]
pair = splits["train"][0] if DO_SPLITS[0] == "train" else splits[DO_SPLITS[0]][0]
want = {"original": pair[0], **{m: f"{pair[0]}_{pair[1]}" for m in DO_METHODS}}

fig, axes = plt.subplots(1, len(show_methods), figsize=(3 * len(show_methods), 3.4))
for ax, m in zip(np.atleast_1d(axes), show_methods):
    ax.axis("off")
    sel = ok[(ok["method"] == m) & (ok["video"] == want[m])]
    if not len(sel):
        sel = ok[ok["method"] == m].head(1)
    if len(sel):
        fp = sorted((FACE_ROOT / sel.iloc[0]["rel"] / "frame").glob(f"*{CROP_EXT}"))
        if fp:
            ax.imshow(Image.open(fp[0]))
            ax.set_title(f"{m}\n{sel.iloc[0]['video']}", fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / "crop_examples.png", dpi=130)
plt.show()

## 7. เขียน split json + แพ็ก output

`Image_dataset` ของ repo มองหา `{root}/FF++/face_v1/splits/{split}.json` → copy ไฟล์ split ไปไว้ให้

ถ้า `SAVE_MODE == "tar"` จะแพ็กเป็น `.tar` แยกตาม method (ไม่บีบอัด เพราะ PNG บีบแล้ว) ลง `/kaggle/working`
ซึ่ง commit เร็วกว่าการเก็บไฟล์ย่อย 40,000 ไฟล์มาก — ระดับ 3b จะแตก tar ลง `/tmp` ก่อนเทรน

In [ ]:
# 1) split json
sp_out = FACE_ROOT / "splits"
sp_out.mkdir(parents=True, exist_ok=True)
for s in ["train", "val", "test"]:
    shutil.copyfile(SPLIT_DIR / f"{s}.json", sp_out / f"{s}.json")

# 2) manifest ไว้ตรวจซ้ำตอนเทรน
manifest = crop_log[["rel", "split", "method", "video", "label", "n_saved"]].copy()
manifest.to_csv(FACE_ROOT / "manifest.csv", index=False)
meta = {"repo_commit": REPO_COMMIT, "compression": COMPRESSION, "n_frames": N_FRAMES,
        "image_size": IMG_SIZE, "det_thres": DET_THRES, "max_shift": MAX_SHIFT,
        "crop_ext": CROP_EXT, "methods": DO_METHODS, "splits": DO_SPLITS,
        "n_videos_ok": int(crop_log["ok"].sum()), "n_frames_total": int(crop_log["n_saved"].sum()),
        "env": ENV, "created": time.strftime("%Y-%m-%d %H:%M:%S")}
json.dump(meta, open(FACE_ROOT / "prepare_meta.json", "w"), indent=2)
print(json.dumps(meta, indent=2)[:600])

# 3) แพ็ก
if SAVE_MODE == "tar":
    tar_dir = WORK_DIR / DATASET_NAME
    tar_dir.mkdir(parents=True, exist_ok=True)
    base = FACE_ROOT.parent.parent          # .../ffpp_c23_faces
    groups = {"meta": [sp_out, FACE_ROOT / "manifest.csv", FACE_ROOT / "prepare_meta.json"],
              "original": [FACE_ROOT / "original_sequences"]}
    for m in DO_METHODS:
        groups[m] = [FACE_ROOT / "manipulated_sequences" / m]
    for name, paths in groups.items():
        paths = [p for p in paths if Path(p).exists()]
        if not paths:
            continue
        tp = tar_dir / f"{name}.tar"
        with tarfile.open(tp, "w") as tar:
            for p in paths:
                tar.add(p, arcname=str(Path(p).relative_to(base)))
        print(f"{tp.name:<22} {tp.stat().st_size/1e9:.2f} GB")
    print("\nขนาดรวมใน /kaggle/working:",
          f"{sum(f.stat().st_size for f in tar_dir.rglob('*') if f.is_file())/1e9:.2f} GB")
else:
    print("crops อยู่ที่", FACE_ROOT)

## 8. ขั้นต่อไป

1. **Save Version → Save & Run All (Commit)** ให้รันจนจบ แล้ว output จะกลายเป็น Kaggle Dataset ของ notebook นี้
2. ในระดับ 3b (fine-tune) ให้ *Add Input* เป็น output ของ notebook นี้
3. ถ้ารันไม่จบใน 12 ชม. ให้แบ่งเป็นหลายรอบด้วย `DO_METHODS` เช่นรอบแรก `["Deepfakes", "Face2Face"]`
   รอบสอง `["FaceSwap", "NeuralTextures"]` แล้ว Add Input ทั้งสอง output ในระดับ 3b

**สรุปสิ่งที่ทำไปแล้ว (เขียนลงรายงานได้เลย)**

In [ ]:
print(f"""
{'='*68}
สรุปการเตรียมข้อมูล FF++ {COMPRESSION}
{'='*68}
repo commit        : {REPO_COMMIT}
split              : official FF++ splits (ondyari/FaceForensics) 360/70/70 identity pairs
                     ตรวจแล้วไม่มี identity ซ้ำข้าม split
frame ต่อวิดีโอ     : {N_FRAMES} (step = N // {N_FRAMES}, เหมือน get_sampled_idx ของ repo)
face detector      : RetinaFace mobilenet0.25 (FTCN), score >= {DET_THRES}
alignment / crop   : 5-point ArcFace alignment -> crop_face_sbi(phase='test') -> {IMG_SIZE}x{IMG_SIZE}
รูปแบบไฟล์         : {CROP_EXT} (lossless)
วิดีโอที่ใช้ได้      : {int(crop_log['ok'].sum()):,} / {len(crop_log):,}
frame ทั้งหมด       : {int(crop_log['n_saved'].sum()):,}
เวลารวม            : {(time.time()-T_START)/60:.1f} นาที
{'='*68}
""")